# Pemeringkatan Influencer: Entropy Weight + EDAS

Disusun per **BLOK** (samaan kayak notebook scoring sebelumnya), tiap blok ada cell bernomor.

| Blok | Isi |
|---|---|
| **A** | Setup, config, load semua sumber data (skor sentimen, followers, metadata video, views, comments) |
| **B** | Hitung C3 (views), C4 (shares), C5 (engagement rate, 2 formula) -- SEMUA dari video yang disentimenin doang |
| **C** | Susun matriks keputusan (gabung C1+C2+C3+C4+C5) + filter followers & threshold komentar |
| **D** | Fungsi Entropy Weight (Persamaan 2.2-2.6 proposal) |
| **E** | Fungsi EDAS (Persamaan 2.7-2.14 proposal) |
| **F** | Jalanin semua kombinasi skenario (3 skema sentimen x 3 threshold komentar x 2 formula ER) + rangking |

**Catatan skenario yang disepakati:**
- Kriteria lengkap sesuai Tabel 3.1 proposal: **C1** (sentimen), **C2** (followers), **C3** (rata-rata views), **C4** (rata-rata shares), **C5** (engagement rate).
- **PENTING -- scope video buat C3/C4/C5**: cuma dihitung dari video_id yang ADA di `comments.csv` (yaitu video yang sama yang dipakai buat ngitung C1/sentimen). Video yang ga disentimenin (ga ada di `comments.csv`) ga ikut dihitung sama sekali di C3/C4/C5, biar semua kriteria video-level ngukur konten yang sama persis.
- C3 (views) tambahan punya sub-scope lebih sempit lagi: cuma video yang BERHASIL ke-scrape views-nya (lihat `video_views.csv`, hasil scraping tambahan) -- ada video yang gagal/ga ketemu, jadi C3 & C5_B (lihat bawah) bisa dihitung dari video lebih sedikit lagi dibanding C1/C4. Ini keterbatasan metodologi yang perlu disebut di skripsi.
- **2 formula engagement rate (C5) yang dibandingin:**
  - `C5_A` = rata-rata total interaksi (like+comment+share+save) / **followers** -- dari semua video yang disentimenin.
  - `C5_B` = rata-rata total interaksi / **views** -- dihitung cuma dari video yang disentimenin DAN berhasil ke-scrape views-nya (biar numerator & denominator dari video yang sama).
- C1 dicoba 3 varian: `3class` (positif=+1/netral=0/negatif=-1), `3class_hanya_positif` (positif=+1/netral=0/negatif=0), `binary` (pake varian B: negatif=-1/non-negatif=+1).
- Threshold minimal komentar bersih per influencer dicoba 3 skenario: **30, 50, 100**.
- Filter tambahan dari batasan masalah proposal: **followers >= 10.000**.
- Filter tambahan lagi: **minimal 10 video disentimenin** per influencer, DAN **minimal 10 video yang juga punya data views** -- berlaku di SEMUA skenario (A maupun B), soalnya C3 (views) sekarang jadi kriteria wajib di matriks buat kedua formula. Filter ini biar C3/C4/C5 ga dihitung dari sample video yang kelewat sedikit (rawan di-drag 1 video viral jadi outlier, seperti kasus `farhanzubedi` yang ketahuan di Cell F5/F6).
- Total: **3 skema x 3 threshold x 2 formula ER = 18 kombinasi EDAS** yang dijalanin & dibandingin.

---
## BLOK A — Setup, Config, Load Data

**Cell A1**: import & konfigurasi (path, daftar skema C1, daftar threshold, min followers).
**Cell A2**: cari folder root repo + susun semua path.
**Cell A3**: load `skor_influencer.csv`, `metadata_profil.csv`, `metadata_video.csv`.

In [1]:
# ==== Cell A1: Import & Konfigurasi ====
import numpy as np
import pandas as pd
from pathlib import Path

EXCLUDE_USERNAMES = ['drrichardlee']   # konsisten sama tahap scoring sebelumnya

# --- 3 skema C1 yang mau dicoba ---
SKEMA_C1 = {
    '3class': 'C1_3class',
    '3class_hanya_positif': 'C1_3class_hanya_positif',
    'binary': 'C1_binary_varianB',
}

# --- 3 threshold minimal komentar bersih per influencer yang mau dicoba ---
DAFTAR_THRESHOLD_KOMENTAR = [30, 50, 100]

# --- Batasan dari proposal (bab 1.3): influencer minimal 10.000 followers ---
MIN_FOLLOWERS = 10_000

# --- Filter tambahan: minimal 10 video (yang disentimenin), biar C3/C4/C5 (dihitung dari ---
# --- rata-rata per video) ga rawan di-drag outlier dari 1-2 video viral doang ---
MIN_VIDEO = 10

# --- Filter khusus skenario C5_B (ER/views): minimal 10 video yang JUGA punya data views ---
MIN_VIDEO_VIEWS = 10

# --- 2 formula engagement rate (C5) yang mau dibandingin ---
DAFTAR_FORMULA_C5 = ['A', 'B']   # A = interaksi/followers, B = interaksi/views

print('Config siap.')
print('Skema C1:', list(SKEMA_C1.keys()))
print('Threshold komentar yg dicoba:', DAFTAR_THRESHOLD_KOMENTAR)
print('Formula C5 yg dicoba:', DAFTAR_FORMULA_C5)
print('Min followers:', MIN_FOLLOWERS)
print('Min jumlah video (disentimenin):', MIN_VIDEO)
print('Min jumlah video (ada data views, buat C5_B):', MIN_VIDEO_VIEWS)


Config siap.
Skema C1: ['3class', '3class_hanya_positif', 'binary']
Threshold komentar yg dicoba: [30, 50, 100]
Formula C5 yg dicoba: ['A', 'B']
Min followers: 10000
Min jumlah video (disentimenin): 10
Min jumlah video (ada data views, buat C5_B): 10


In [2]:
# ==== Cell A2: Cari folder root repo + path ====
def find_repo_root():
    current = Path.cwd()
    for _ in range(6):
        if (current / '01_pengumpulan_data').is_dir() and (current / '03_Indobert').is_dir():
            return current
        if current.parent == current:
            break
        current = current.parent
    return None


REPO_ROOT = find_repo_root()
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Nggak ketemu folder root repo (butuh '01_pengumpulan_data' dan '03_Indobert' "
        "keduanya ada sebagai subfolder). Jalankan dari dalam folder repo."
    )
print('Repo root:', REPO_ROOT)

PATH_SKOR_INFLUENCER = REPO_ROOT / '03_Indobert' / 'Data' / 'hasil_scoring' / 'skor_influencer.csv'
PATH_PROFIL = REPO_ROOT / '01_pengumpulan_data' / '04_1_mendapatkan_metadata_akun' / 'metadata_profil.csv'
PATH_VIDEO = REPO_ROOT / '01_pengumpulan_data' / '04_1_mendapatkan_metadata_akun' / 'metadata_video.csv'
PATH_VIEWS = REPO_ROOT / '01_pengumpulan_data' / '04_1_mendapatkan_metadata_akun' / 'video_views.csv'
PATH_COMMENTS = REPO_ROOT / '01_pengumpulan_data' / '05_scrap_komentar' / 'hasil_07_comments_01' / 'comments.csv'
OUT_DIR = REPO_ROOT / '03_Indobert' / 'Data' / 'hasil_edas'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Skor influencer :', PATH_SKOR_INFLUENCER)
print('Profil (followers):', PATH_PROFIL)
print('Video (shares dkk):', PATH_VIDEO)
print('Views (tambahan)  :', PATH_VIEWS)
print('Comments (buat tau video mana yg disentimenin):', PATH_COMMENTS)
print('Output dir      :', OUT_DIR)


Repo root: c:\Users\nesha\Tugas_akhir\seleksi-influencer
Skor influencer : c:\Users\nesha\Tugas_akhir\seleksi-influencer\03_Indobert\Data\hasil_scoring\skor_influencer.csv
Profil (followers): c:\Users\nesha\Tugas_akhir\seleksi-influencer\01_pengumpulan_data\04_1_mendapatkan_metadata_akun\metadata_profil.csv
Video (shares dkk): c:\Users\nesha\Tugas_akhir\seleksi-influencer\01_pengumpulan_data\04_1_mendapatkan_metadata_akun\metadata_video.csv
Views (tambahan)  : c:\Users\nesha\Tugas_akhir\seleksi-influencer\01_pengumpulan_data\04_1_mendapatkan_metadata_akun\video_views.csv
Comments (buat tau video mana yg disentimenin): c:\Users\nesha\Tugas_akhir\seleksi-influencer\01_pengumpulan_data\05_scrap_komentar\hasil_07_comments_01\comments.csv
Output dir      : c:\Users\nesha\Tugas_akhir\seleksi-influencer\03_Indobert\Data\hasil_edas


In [3]:
# ==== Cell A3: Load semua sumber data ====
skor_influencer = pd.read_csv(PATH_SKOR_INFLUENCER, dtype=str, keep_default_na=False, encoding='utf-8')
# kolom numerik di skor_influencer perlu di-convert balik (tadinya disimpen sbg string biar aman)
for kol in skor_influencer.columns:
    if kol != 'username':
        skor_influencer[kol] = pd.to_numeric(skor_influencer[kol], errors='coerce')

df_profil = pd.read_csv(PATH_PROFIL, dtype=str, keep_default_na=False, encoding='utf-8')
df_video = pd.read_csv(PATH_VIDEO, dtype=str, keep_default_na=False, encoding='utf-8')
df_views = pd.read_csv(PATH_VIEWS, dtype=str, keep_default_na=False, encoding='utf-8')
df_comments = pd.read_csv(PATH_COMMENTS, dtype=str, keep_default_na=False, encoding='utf-8', usecols=['video_id'])

for nama, df, req in [
    ('skor_influencer.csv', skor_influencer, {'username'} | set(SKEMA_C1.values())),
    ('metadata_profil.csv', df_profil, {'username', 'followers', 'scraped_at'}),
    ('metadata_video.csv', df_video, {'username', 'video_id', 'like_count', 'comment_count', 'share_count', 'save_count'}),
    ('video_views.csv', df_views, {'video_id', 'views'}),
    ('comments.csv', df_comments, {'video_id'}),
]:
    missing = req - set(df.columns)
    if missing:
        raise ValueError(f'{nama} kurang kolom: {missing}')

print(f'skor_influencer : {len(skor_influencer):,} influencer')
print(f'metadata_profil : {len(df_profil):,} baris')
print(f'metadata_video  : {len(df_video):,} baris')
print(f'video_views     : {len(df_views):,} baris')
print(f'comments        : {len(df_comments):,} baris (buat ambil video_id unik yg disentimenin)')


skor_influencer : 42 influencer
metadata_profil : 47 baris
metadata_video  : 6,939 baris
video_views     : 628 baris
comments        : 24,463 baris (buat ambil video_id unik yg disentimenin)


---
## BLOK B — Hitung C3 (views), C4 (shares), C5 (engagement rate, 2 formula)

**Semua dari video yang disentimenin doang** (video_id yang ada di `comments.csv`) -- video yang
ga disentimenin ga ikut dihitung sama sekali di C3/C4/C5.

**Cell B1**: ambil followers per influencer (dedup kalau ada baris duplikat, ambil yang paling baru).
**Cell B2**: ambil `video_id` unik yang disentimenin (dari `comments.csv`), filter `metadata_video.csv`
cuma ke video_id itu, lalu agregasi -> C4 (rata-rata shares) & rata-rata total interaksi per influencer.
**Cell B3**: gabungin lagi sama `video_views.csv` (cuma video yang disentimenin DAN berhasil ke-scrape
views-nya) -> C3 (rata-rata views) & rata-rata total interaksi versi subset-views (buat numerator C5_B).
**Cell B4**: gabungin semua jadi `metrik_akun`, hitung **C5_A** (interaksi/followers) & **C5_B** (interaksi/views).

In [4]:
# ==== Cell B1: Ambil followers per influencer ====
def ambil_followers(df_profil, exclude_usernames):
    df = df_profil[~df_profil['username'].isin(exclude_usernames)].copy()
    df['followers'] = pd.to_numeric(df['followers'], errors='coerce')
    # kalau 1 username muncul >1 baris (di-scrape ulang beberapa kali), ambil yg paling baru
    df = df.sort_values('scraped_at').drop_duplicates(subset='username', keep='last')
    return df[['username', 'followers']].reset_index(drop=True)


followers_df = ambil_followers(df_profil, EXCLUDE_USERNAMES)
print(f'Followers ter-load: {len(followers_df)} influencer')
followers_df.sort_values('followers', ascending=False).head(10)


Followers ter-load: 46 influencer


,username,followers
34,\drrichardlee,6400000.0
39,drkevinmak,4200000.0
1,tirtacipeng,2800000.0
27,dokter_ika,2600000.0
40,farhanzubedi,2500000.0
28,dokteramiraobgyn,2300000.0
24,aymanalts,2000000.0
32,dr.ziee,1800000.0
14,ekidarehanf,1400000.0
6,nadialaydrus,1400000.0


In [5]:
# ==== Cell B2: Video yang DISENTIMENIN doang -> C4 & numerator C5_A ====
video_id_disentimenkan = set(df_comments['video_id'].dropna().unique())
print(f'Jumlah video_id unik yang disentimenin (ada di comments.csv): {len(video_id_disentimenkan)}')

df_video_sentimen = df_video[df_video['video_id'].isin(video_id_disentimenkan)].copy()
print(f'metadata_video.csv: {len(df_video)} baris total -> {len(df_video_sentimen)} baris '
      f'yang video_id-nya disentimenin ({len(df_video) - len(df_video_sentimen)} baris dibuang).')


def agregasi_video(df_video_sentimen, exclude_usernames):
    df = df_video_sentimen[~df_video_sentimen['username'].isin(exclude_usernames)].copy()
    for kol in ['like_count', 'comment_count', 'share_count', 'save_count']:
        df[kol] = pd.to_numeric(df[kol], errors='coerce').fillna(0)
    df['total_interaksi'] = df['like_count'] + df['comment_count'] + df['share_count'] + df['save_count']
    agg = df.groupby('username').agg(
        C4_rata_shares=('share_count', 'mean'),
        rata_total_interaksi=('total_interaksi', 'mean'),
        jumlah_video=('video_id', 'nunique'),
    ).reset_index()
    return agg


video_agg = agregasi_video(df_video_sentimen, EXCLUDE_USERNAMES)
print(f'Video agg (disentimenin) ter-load: {len(video_agg)} influencer')
video_agg.sort_values('C4_rata_shares', ascending=False).head(10)


Jumlah video_id unik yang disentimenin (ada di comments.csv): 656
metadata_video.csv: 6939 baris total -> 656 baris yang video_id-nya disentimenin (6283 baris dibuang).
Video agg (disentimenin) ter-load: 42 influencer


,username,C4_rata_shares,rata_total_interaksi,jumlah_video
24,farhanzubedi,75130.666667,422153.000000,3
29,ikhsanqthii,10042.750000,68579.400000,20
27,giapratamamd,3734.250000,48988.100000,20
39,tirtacipeng,3654.263158,51830.842105,19
9,doktermedok,2133.200000,11517.400000,20
36,putriemje,1890.285714,4249.714286,7
31,jamilatussadiyah,1807.352941,11616.117647,17
23,ekidarehanf,1202.950000,16424.950000,20
14,dr.mariojohan,1078.421053,11050.052632,19
21,drkevinmak,1027.263158,6758.526316,19


In [6]:
# ==== Cell B3: Gabung sama video_views.csv -> C3 & numerator C5_B ====
# df_views cuma nyakup video_id yang disentimenin JUGA (scraping views di-scope ke situ), tapi
# ga semuanya berhasil ke-scrape -- makanya di-INNER JOIN lagi, subset-nya lebih sempit lagi.
df_views_num = df_views.copy()
df_views_num['views'] = pd.to_numeric(df_views_num['views'], errors='coerce')
df_views_num = df_views_num.dropna(subset=['views'])

df_video_sentimen_views = df_video_sentimen.merge(
    df_views_num[['video_id', 'views']], on='video_id', how='inner')
print(f'Video disentimenin: {len(df_video_sentimen)} baris -> {len(df_video_sentimen_views)} baris '
      f'yang JUGA berhasil ke-scrape views-nya.')

df_video_sentimen_views = df_video_sentimen_views.copy()
for kol in ['like_count', 'comment_count', 'share_count', 'save_count']:
    df_video_sentimen_views[kol] = pd.to_numeric(df_video_sentimen_views[kol], errors='coerce').fillna(0)
df_video_sentimen_views['total_interaksi'] = (
    df_video_sentimen_views['like_count'] + df_video_sentimen_views['comment_count']
    + df_video_sentimen_views['share_count'] + df_video_sentimen_views['save_count']
)

video_agg_views = df_video_sentimen_views.groupby('username').agg(
    C3_rata_views=('views', 'mean'),
    rata_total_interaksi_views=('total_interaksi', 'mean'),
    jumlah_video_views=('video_id', 'nunique'),
).reset_index()

print(f'Video agg (disentimenin + ada views) ter-load: {len(video_agg_views)} influencer')
video_agg_views.sort_values('C3_rata_views', ascending=False).head(10)


Video disentimenin: 656 baris -> 170 baris yang JUGA berhasil ke-scrape views-nya.
Video agg (disentimenin + ada views) ter-load: 27 influencer


,username,C3_rata_views,rata_total_interaksi_views,jumlah_video_views
15,ekidarehanf,9096.000000,189.000000,1
13,drkevinmak,8097.000000,164.000000,1
0,dionharyadi,7522.750000,333.500000,4
20,jiemiardian,6714.200000,660.200000,5
25,roozzaak,6369.750000,225.500000,4
10,dr.zulvia.syarif.spkj,6226.333333,226.333333,3
3,doktermatahendra,5731.428571,138.142857,7
7,dr.mariojohan,5368.800000,210.800000,5
18,itsdoktermuda,5304.800000,138.200000,5
4,dokterprasadja,5141.909091,185.909091,11


In [7]:
# ==== Cell B4: Gabungin semua jadi metrik_akun, hitung C5_A & C5_B ====
metrik_akun = followers_df.merge(video_agg, on='username', how='inner')
metrik_akun = metrik_akun.merge(video_agg_views, on='username', how='left')
# jumlah_video_views NaN (ga ada 1 pun video yang kescrape views-nya) -> anggap 0, bukan dibuang
metrik_akun['jumlah_video_views'] = metrik_akun['jumlah_video_views'].fillna(0)

metrik_akun['C5_A_engagement_rate'] = metrik_akun['rata_total_interaksi'] / metrik_akun['followers']
metrik_akun['C5_B_engagement_rate'] = metrik_akun['rata_total_interaksi_views'] / metrik_akun['C3_rata_views']

print(f'Metrik akun lengkap: {len(metrik_akun)} influencer '
      f'(irisan yg ada DI followers.csv MAUPUN video.csv)')
print(f'Dari situ, yang punya minimal 1 video dgn data views (bisa dihitung C3/C5_B): '
      f'{(metrik_akun["jumlah_video_views"] > 0).sum()} influencer')
metrik_akun.sort_values('C5_A_engagement_rate', ascending=False).head(10)


Metrik akun lengkap: 42 influencer (irisan yg ada DI followers.csv MAUPUN video.csv)
Dari situ, yang punya minimal 1 video dgn data views (bisa dihitung C3/C5_B): 27 influencer


,username,followers,C4_rata_shares,rata_total_interaksi,jumlah_video,C3_rata_views,rata_total_interaksi_views,jumlah_video_views,C5_A_engagement_rate,C5_B_engagement_rate
36,farhanzubedi,2500000.0,75130.666667,422153.000000,3,NaN,NaN,0.0,0.168861,NaN
16,fauzaannibras,13200.0,74.250000,931.500000,12,4496.000000,66.000000,5.0,0.070568,0.014680
39,ikhsanqthii,974300.0,10042.750000,68579.400000,20,NaN,NaN,0.0,0.070388,NaN
40,jamilatussadiyah,202500.0,1807.352941,11616.117647,17,4234.500000,66.500000,2.0,0.057364,0.015704
32,dokter.harmonospanak,19500.0,131.684211,881.368421,19,4380.357143,129.285714,14.0,0.045198,0.029515
38,giapratamamd,1300000.0,3734.250000,48988.100000,20,NaN,NaN,0.0,0.037683,NaN
30,drgfira,21700.0,18.857143,682.714286,7,5007.000000,193.750000,4.0,0.031461,0.038696
28,dr.deamaulidia,29400.0,50.222222,917.777778,9,2870.000000,24.400000,5.0,0.031217,0.008502
37,gelarsramdhani,12800.0,48.411765,382.117647,17,2985.555556,72.333333,9.0,0.029853,0.024228
20,dokterprasadja,48500.0,156.500000,1052.875000,16,5141.909091,185.909091,11.0,0.021709,0.036156


---
## BLOK C — Susun Matriks Keputusan + Filter

**Cell C1**: fungsi `susun_matriks()` -- gabungin C1 (sesuai skema yg dipilih) + C2 + C3 + C4 + C5
(sesuai formula ER yg dipilih: A atau B), filter followers >= 10.000, jumlah video disentimenin >= 10,
jumlah komentar >= threshold, dan jumlah video yg ada views-nya >= 10 (berlaku di formula A maupun B,
soalnya C3 wajib ada di matriks buat keduanya).
**Cell C2**: contoh coba 1 kombinasi (skema `3class`, threshold 30, formula `A`) buat ngecek matriksnya kebentuk bener.
**Cell C3**: kombinasi yang sama tapi formula `B` -- angka yg kefilter harusnya SAMA kayak formula A
(filter video-views-nya udah berlaku buat dua-duanya), bedanya cuma di nilai C5.

In [8]:
# ==== Cell C1: Fungsi susun matriks keputusan ====
def susun_matriks(skor_influencer_df, nama_skema, metrik_akun_df, formula_c5='A',
                   min_followers=MIN_FOLLOWERS, min_komentar=30, min_video=MIN_VIDEO,
                   min_video_views=MIN_VIDEO_VIEWS):
    """Gabungin C1 (sesuai skema) + C2/C3/C4/C5 dari metrik_akun, lalu filter followers, jumlah
    video, jumlah komentar, dan (kalau formula_c5='B') jumlah video yg ada data views-nya.
    Return: df matriks (index=username, kolom C1/C2/C3/C4/C5) + jumlah yg kefilter.

    Filter jumlah video penting karena C3/C4/C5 dihitung sbg RATA-RATA per video -- kalau cuma
    dihitung dari sedikit video, rata-ratanya rawan di-drag jauh sama 1 video yg kebetulan
    viral (lihat kasus farhanzubedi di Cell F5/F6). formula_c5='B' butuh filter tambahan karena
    C3/C5_B dihitung dari subset video yg lebih sempit lagi (yg berhasil ke-scrape views-nya)."""
    kolom_c1 = SKEMA_C1[nama_skema]
    kolom_p = 'P_' + kolom_c1.replace('C1_', '')
    kolom_c5 = 'C5_A_engagement_rate' if formula_c5 == 'A' else 'C5_B_engagement_rate'

    df = skor_influencer_df[['username', kolom_c1, kolom_p]].merge(
        metrik_akun_df, on='username', how='inner')

    sebelum = len(df)
    df = df[df['followers'] >= min_followers]
    df = df[df['jumlah_video'] >= min_video]
    df = df[df[kolom_p] >= min_komentar]
    # C3 (views) WAJIB ada di matriks buat formula A maupun B -- jadi filter video-views ini
    # selalu berlaku, ga cuma pas formula_c5='B'. Kalau ga, C3 bisa NaN & bikin AS/rank error.
    df = df[df['jumlah_video_views'] >= min_video_views]
    df = df[df['C3_rata_views'].notna()]
    df = df[df[kolom_c5].notna()]
    sesudah = len(df)

    matriks = df.rename(columns={
        kolom_c1: 'C1', 'followers': 'C2', 'C3_rata_views': 'C3',
        'C4_rata_shares': 'C4', kolom_c5: 'C5',
    }).set_index('username')[['C1', 'C2', 'C3', 'C4', 'C5']]

    return matriks, sebelum, sesudah


print('Fungsi susun_matriks() siap.')


Fungsi susun_matriks() siap.


In [9]:
# ==== Cell C2: Coba 1 kombinasi dulu (skema 3class, threshold 30, formula C5_A) ====
matriks_contoh, sebelum, sesudah = susun_matriks(
    skor_influencer, '3class', metrik_akun, formula_c5='A',
    min_followers=MIN_FOLLOWERS, min_komentar=30, min_video=MIN_VIDEO)
print(f'Influencer sebelum filter: {sebelum}, sesudah filter (followers>={MIN_FOLLOWERS}, '
      f'video>={MIN_VIDEO}, komentar>=30, formula=A): {sesudah}')
matriks_contoh.sort_values('C1', ascending=False)


Influencer sebelum filter: 42, sesudah filter (followers>=10000, video>=10, komentar>=30, formula=A): 4


,C1,C2,C3,C4,C5
username,,,,,
dokter.harmonospanak,0.326241,19500.0,4380.357143,131.684211,0.045198
meta.hanindita,0.112903,68000.0,4735.090909,4.461538,0.001640
zickyneuro,0.023256,256600.0,2506.400000,41.692308,0.001610
dokterprasadja,0.020101,48500.0,5141.909091,156.500000,0.021709


In [10]:
# ==== Cell C3: Kombinasi yang sama, tapi formula C5_B (ER/views) ====
matriks_contoh_B, sebelum_B, sesudah_B = susun_matriks(
    skor_influencer, '3class', metrik_akun, formula_c5='B',
    min_followers=MIN_FOLLOWERS, min_komentar=30, min_video=MIN_VIDEO,
    min_video_views=MIN_VIDEO_VIEWS)
print(f'Influencer sebelum filter: {sebelum_B}, sesudah filter (followers>={MIN_FOLLOWERS}, '
      f'video>={MIN_VIDEO}, komentar>=30, video_views>={MIN_VIDEO_VIEWS}, formula=B): {sesudah_B}')
print(f'(Bandingin: formula A lolos {sesudah} influencer, formula B lolos {sesudah_B} influencer)')
matriks_contoh_B.sort_values('C1', ascending=False)


Influencer sebelum filter: 42, sesudah filter (followers>=10000, video>=10, komentar>=30, video_views>=10, formula=B): 4
(Bandingin: formula A lolos 4 influencer, formula B lolos 4 influencer)


,C1,C2,C3,C4,C5
username,,,,,
dokter.harmonospanak,0.326241,19500.0,4380.357143,131.684211,0.029515
meta.hanindita,0.112903,68000.0,4735.090909,4.461538,0.020063
zickyneuro,0.023256,256600.0,2506.400000,41.692308,0.033674
dokterprasadja,0.020101,48500.0,5141.909091,156.500000,0.036156


---
## BLOK D — Entropy Weight (Persamaan 2.2 - 2.6 proposal)

**Cell D1**: fungsi `hitung_bobot_entropy()` -- normalisasi probabilitas, hitung entropy $E_j$, derajat diversifikasi $d_j$, bobot $w_j$.

In [11]:
# ==== Cell D1: Fungsi Entropy Weight ====
def hitung_bobot_entropy(matriks, kolom_kriteria):
    """Persamaan 2.2 (normalisasi pij), 2.3 (entropy Ej), 2.4 (konstanta k), 2.5 (diversifikasi
    dj), 2.6 (bobot wj). Kalau pij=0, pij*ln(pij) dianggap 0 (sesuai catatan di proposal)."""
    m = len(matriks)
    k = 1 / np.log(m)

    detail = []
    for kol in kolom_kriteria:
        x = matriks[kol].values.astype(float)
        p = x / x.sum()
        p_lnp = np.where(p > 0, p * np.log(p), 0.0)
        Ej = -k * p_lnp.sum()
        dj = 1 - Ej
        detail.append({'kriteria': kol, 'Ej': Ej, 'dj': dj})

    total_d = sum(d['dj'] for d in detail)
    bobot = {}
    for d in detail:
        d['wj'] = d['dj'] / total_d
        bobot[d['kriteria']] = d['wj']

    return bobot, pd.DataFrame(detail).set_index('kriteria')


print('Fungsi hitung_bobot_entropy() siap.')

# Coba langsung ke matriks contoh dari Cell C2
bobot_contoh, detail_entropy_contoh = hitung_bobot_entropy(matriks_contoh, ['C1', 'C2', 'C3', 'C4', 'C5'])
print('\nBobot entropy (contoh, skema 3class, threshold 30, formula C5_A):')
print(detail_entropy_contoh.to_string())
print('\nTotal bobot (harus = 1.0):', sum(bobot_contoh.values()))


Fungsi hitung_bobot_entropy() siap.

Bobot entropy (contoh, skema 3class, threshold 30, formula C5_A):
                Ej        dj        wj
kriteria                              
C1        0.636975  0.363025  0.272805
C2        0.713482  0.286518  0.215312
C3        0.976991  0.023009  0.017291
C4        0.749854  0.250146  0.187979
C5        0.591984  0.408016  0.306614

Total bobot (harus = 1.0): 1.0


---
## BLOK E — EDAS (Persamaan 2.7 - 2.14 proposal)

**Cell E1**: fungsi `hitung_edas()` -- rata-rata kriteria (AV), jarak positif/negatif dari rata-rata (PDA/NDA), skor berbobot (SP/SN), normalisasi (NSP/NSN), skor akhir (Appraisement Score / AS) + rangking.

In [12]:
# ==== Cell E1: Fungsi EDAS ====
def hitung_edas(matriks, kolom_kriteria, bobot):
    """Semua kriteria di penelitian ini tipe BENEFIT (sesuai Tabel 3.1 proposal), jadi cuma
    pakai rumus PDA/NDA versi benefit (Persamaan 2.8 & 2.9)."""
    df = matriks.copy()
    AV = {kol: df[kol].mean() for kol in kolom_kriteria}

    for kol in kolom_kriteria:
        avj = AV[kol]
        df[f'PDA_{kol}'] = np.maximum(0, df[kol] - avj) / avj
        df[f'NDA_{kol}'] = np.maximum(0, avj - df[kol]) / avj

    df['SP'] = sum(bobot[kol] * df[f'PDA_{kol}'] for kol in kolom_kriteria)
    df['SN'] = sum(bobot[kol] * df[f'NDA_{kol}'] for kol in kolom_kriteria)

    max_SP, max_SN = df['SP'].max(), df['SN'].max()
    df['NSP'] = df['SP'] / max_SP if max_SP != 0 else 0.0
    df['NSN'] = 1 - (df['SN'] / max_SN if max_SN != 0 else 0.0)

    df['AS'] = 0.5 * (df['NSP'] + df['NSN'])
    df['rank'] = df['AS'].rank(ascending=False, method='min').astype(int)
    return df.sort_values('AS', ascending=False)


print('Fungsi hitung_edas() siap.')

# Coba langsung ke matriks & bobot contoh
hasil_contoh = hitung_edas(matriks_contoh, ['C1', 'C2', 'C3', 'C4', 'C5'], bobot_contoh)
print('\nHasil EDAS (contoh, skema 3class, threshold 30, formula C5_A), top 10:')
hasil_contoh[['C1', 'C2', 'C3', 'C4', 'C5', 'SP', 'SN', 'AS', 'rank']].head(10)


Fungsi hitung_edas() siap.

Hasil EDAS (contoh, skema 3class, threshold 30, formula C5_A), top 10:


,C1,C2,C3,C4,C5,SP,SN,AS,rank
username,,,,,,,,,
dokter.harmonospanak,0.326241,19500.0,4380.357143,131.684211,0.045198,1.057495,0.172534,0.856183,1
dokterprasadja,0.020101,48500.0,5141.909091,156.500000,0.021709,0.240795,0.336263,0.333559,2
zickyneuro,0.023256,256600.0,2506.400000,41.692308,0.001610,0.347592,0.599841,0.164347,3
meta.hanindita,0.112903,68000.0,4735.090909,4.461538,0.001640,0.002245,0.539489,0.051368,4


---
## BLOK F — Jalanin Semua Kombinasi (3 skema x 3 threshold x 2 formula C5) + Bandingin

**Cell F1**: loop semua kombinasi (18 total), simpan hasil tiap kombinasi.
**Cell F2**: simpan semua hasil ke CSV.
**Cell F3**: tampilin ringkasan top-10 tiap kombinasi.
**Cell F4**: bandingin rangking antar kombinasi pake korelasi Spearman (dasar buat sensitivity analysis di proposal, bab 3.1.7).

In [13]:
# ==== Cell F1: Loop semua kombinasi (3 skema x 3 threshold x 2 formula C5) ====
semua_hasil = {}
ringkasan_kombinasi = []

for nama_skema in SKEMA_C1:
    for threshold in DAFTAR_THRESHOLD_KOMENTAR:
        for formula_c5 in DAFTAR_FORMULA_C5:
            matriks, sebelum, sesudah = susun_matriks(
                skor_influencer, nama_skema, metrik_akun, formula_c5=formula_c5,
                min_followers=MIN_FOLLOWERS, min_komentar=threshold, min_video=MIN_VIDEO,
                min_video_views=MIN_VIDEO_VIEWS,
            )
            bobot, detail_entropy = hitung_bobot_entropy(matriks, ['C1', 'C2', 'C3', 'C4', 'C5'])
            hasil = hitung_edas(matriks, ['C1', 'C2', 'C3', 'C4', 'C5'], bobot)

            key = f'{nama_skema}__min{threshold}__C5{formula_c5}'
            semua_hasil[key] = {'matriks': matriks, 'bobot': bobot, 'detail_entropy': detail_entropy, 'hasil': hasil}
            ringkasan_kombinasi.append({
                'skema': nama_skema, 'threshold_komentar': threshold, 'formula_c5': formula_c5,
                'jumlah_influencer': sesudah, 'top1': hasil.index[0], 'top1_AS': hasil.iloc[0]['AS'],
            })
            print(f'[{key}] {sesudah} influencer -- top1: {hasil.index[0]} (AS={hasil.iloc[0]["AS"]:.3f})')

ringkasan_df = pd.DataFrame(ringkasan_kombinasi)
print('\nRingkasan semua kombinasi:')
ringkasan_df


[3class__min30__C5A] 4 influencer -- top1: dokter.harmonospanak (AS=0.856)
[3class__min30__C5B] 4 influencer -- top1: dokter.harmonospanak (AS=0.743)
[3class__min50__C5A] 4 influencer -- top1: dokter.harmonospanak (AS=0.856)
[3class__min50__C5B] 4 influencer -- top1: dokter.harmonospanak (AS=0.743)
[3class__min100__C5A] 3 influencer -- top1: dokter.harmonospanak (AS=0.805)
[3class__min100__C5B] 3 influencer -- top1: dokter.harmonospanak (AS=0.765)
[3class_hanya_positif__min30__C5A] 4 influencer -- top1: dokter.harmonospanak (AS=0.845)
[3class_hanya_positif__min30__C5B] 4 influencer -- top1: zickyneuro (AS=0.754)
[3class_hanya_positif__min50__C5A] 4 influencer -- top1: dokter.harmonospanak (AS=0.845)
[3class_hanya_positif__min50__C5B] 4 influencer -- top1: zickyneuro (AS=0.754)
[3class_hanya_positif__min100__C5A] 3 influencer -- top1: dokter.harmonospanak (AS=0.643)
[3class_hanya_positif__min100__C5B] 3 influencer -- top1: zickyneuro (AS=0.789)
[binary__min30__C5A] 4 influencer -- top1:

,skema,threshold_komentar,formula_c5,jumlah_influencer,top1,top1_AS
0,3class,30,A,4,dokter.harmonospanak,0.856183
1,3class,30,B,4,dokter.harmonospanak,0.743249
2,3class,50,A,4,dokter.harmonospanak,0.856183
3,3class,50,B,4,dokter.harmonospanak,0.743249
4,3class,100,A,3,dokter.harmonospanak,0.804834
5,3class,100,B,3,dokter.harmonospanak,0.764986
6,3class_hanya_positif,30,A,4,dokter.harmonospanak,0.844793
7,3class_hanya_positif,30,B,4,zickyneuro,0.754389
8,3class_hanya_positif,50,A,4,dokter.harmonospanak,0.844793
9,3class_hanya_positif,50,B,4,zickyneuro,0.754389


In [14]:
# ==== Cell F2: Simpan semua hasil ke CSV ====
for key, data in semua_hasil.items():
    data['hasil'].to_csv(OUT_DIR / f'edas_{key}.csv', encoding='utf-8')

ringkasan_df.to_csv(OUT_DIR / 'ringkasan_semua_kombinasi.csv', index=False, encoding='utf-8')
print(f'Tersimpan {len(semua_hasil)} file edas_<skema>__min<threshold>__C5<formula>.csv ke {OUT_DIR}')
print(f'Plus ringkasan_semua_kombinasi.csv')


Tersimpan 18 file edas_<skema>__min<threshold>__C5<formula>.csv ke c:\Users\nesha\Tugas_akhir\seleksi-influencer\03_Indobert\Data\hasil_edas
Plus ringkasan_semua_kombinasi.csv


In [15]:
# ==== Cell F3: Tampilin top-10 tiap kombinasi ====
for key, data in semua_hasil.items():
    print(f'\n=== {key} ===')
    print(data['hasil'][['C1', 'C2', 'C3', 'C4', 'C5', 'AS', 'rank']].head(10).to_string())



=== 3class__min30__C5A ===
                            C1        C2           C3          C4        C5        AS  rank
username                                                                                   
dokter.harmonospanak  0.326241   19500.0  4380.357143  131.684211  0.045198  0.856183     1
dokterprasadja        0.020101   48500.0  5141.909091  156.500000  0.021709  0.333559     2
zickyneuro            0.023256  256600.0  2506.400000   41.692308  0.001610  0.164347     3
meta.hanindita        0.112903   68000.0  4735.090909    4.461538  0.001640  0.051368     4

=== 3class__min30__C5B ===
                            C1        C2           C3          C4        C5        AS  rank
username                                                                                   
dokter.harmonospanak  0.326241   19500.0  4380.357143  131.684211  0.029515  0.743249     1
zickyneuro            0.023256  256600.0  2506.400000   41.692308  0.033674  0.326287     2
dokterprasadja        0.

In [16]:
# ==== Cell F4: Bandingin stabilitas rangking antar kombinasi (Spearman) ====
# Gabungin kolom 'rank' dari semua kombinasi jadi 1 tabel (index=username)
tabel_rank = None
for key, data in semua_hasil.items():
    kolom_rank = data['hasil'][['rank']].rename(columns={'rank': key})
    tabel_rank = kolom_rank if tabel_rank is None else tabel_rank.join(kolom_rank, how='outer')

print('Tabel rank tiap influencer, per kombinasi (NaN = influencer kefilter di kombinasi itu):')
print(tabel_rank.to_string())

print('\nKorelasi rank Spearman antar kombinasi (makin deket 1.0 = makin konsisten/stabil):')
korelasi_rank = tabel_rank.corr(method='spearman')
print(korelasi_rank.round(3).to_string())


Tabel rank tiap influencer, per kombinasi (NaN = influencer kefilter di kombinasi itu):
                      3class__min30__C5A  3class__min30__C5B  3class__min50__C5A  3class__min50__C5B  3class__min100__C5A  3class__min100__C5B  3class_hanya_positif__min30__C5A  3class_hanya_positif__min30__C5B  3class_hanya_positif__min50__C5A  3class_hanya_positif__min50__C5B  3class_hanya_positif__min100__C5A  3class_hanya_positif__min100__C5B  binary__min30__C5A  binary__min30__C5B  binary__min50__C5A  binary__min50__C5B  binary__min100__C5A  binary__min100__C5B
username                                                                                                                                                                                                                                                                                                                                                                                                                                                 

---
### Cell F5 & F6 -- Diagnosa: kenapa akun yg postingannya dikit bisa rank 1?

Dipicu pertanyaan: kalau diperhatikan, C1 (sentimen) `farhanzubedi` di beberapa skema malah
mendekati 0 / negatif, tapi dia tetep rank 1 di semua kombinasi. **Cell F5** nampilin tabel
bobot entropy ($w_j$) tiap kombinasi -- biar ketahuan kriteria mana yg paling "berbobot" di
tiap skenario (entropy ngasih bobot besar ke kriteria yang datanya paling tersebar/variatif,
jadi kalau C4 atau C5 datanya tersebar jauh karena ada outlier, bobotnya bisa jadi gede dan
outlier itu jadi dominan ke skor akhir). **Cell F6** bedah kontribusi tiap kriteria ke SP/SN
nya 1 username spesifik, buat ngonfirmasi kriteria mana yg paling nge-drag dia naik.

In [17]:
# ==== Cell F5: Tabel bobot entropy (wj) tiap kombinasi ====
tabel_bobot = pd.DataFrame({key: data['bobot'] for key, data in semua_hasil.items()}).T
tabel_bobot.index.name = 'kombinasi'
print('Bobot entropy (wj) per kriteria, tiap kombinasi:')
print(tabel_bobot.round(4).to_string())

print('\nKriteria dengan bobot terbesar per kombinasi:')
for key in tabel_bobot.index:
    kriteria_max = tabel_bobot.loc[key].idxmax()
    print(f'  {key:32s}: {kriteria_max} (wj={tabel_bobot.loc[key, kriteria_max]:.4f})')


Bobot entropy (wj) per kriteria, tiap kombinasi:
                                       C1      C2      C3      C4      C5
kombinasi                                                                
3class__min30__C5A                 0.2728  0.2153  0.0173  0.1880  0.3066
3class__min30__C5B                 0.3867  0.3052  0.0245  0.2664  0.0172
3class__min50__C5A                 0.2728  0.2153  0.0173  0.1880  0.3066
3class__min50__C5B                 0.3867  0.3052  0.0245  0.2664  0.0172
3class__min100__C5A                0.3988  0.2795  0.0247  0.0710  0.2260
3class__min100__C5B                0.5138  0.3601  0.0318  0.0915  0.0027
3class_hanya_positif__min30__C5A   0.1457  0.2529  0.0203  0.2208  0.3602
3class_hanya_positif__min30__C5B   0.2228  0.3867  0.0311  0.3376  0.0218
3class_hanya_positif__min50__C5A   0.1457  0.2529  0.0203  0.2208  0.3602
3class_hanya_positif__min50__C5B   0.2228  0.3867  0.0311  0.3376  0.0218
3class_hanya_positif__min100__C5A  0.2104  0.3670  0.0324  0.09

In [18]:
# ==== Cell F6: Bedah kontribusi tiap kriteria ke SP/SN untuk 1 username tertentu ====
def bedah_kontribusi(key, username, kolom_kriteria=['C1', 'C2', 'C3', 'C4', 'C5']):
    """Nunjukin nilai mentah, PDA, NDA, dan sumbangan ke SP (bobot*PDA) tiap kriteria buat
    1 username di 1 kombinasi -- buat nunjukin kriteria mana yg paling 'ngangkat' dia."""
    data = semua_hasil[key]
    hasil, bobot = data['hasil'], data['bobot']
    if username not in hasil.index:
        print(f'{username} tidak ada di kombinasi {key} (mungkin kefilter).')
        return None
    baris = hasil.loc[username]
    rincian = []
    for kol in kolom_kriteria:
        rincian.append({
            'kriteria': kol,
            'nilai': baris[kol],
            'wj': bobot[kol],
            'PDA': baris[f'PDA_{kol}'],
            'NDA': baris[f'NDA_{kol}'],
            'sumbangan_SP': bobot[kol] * baris[f'PDA_{kol}'],
            'sumbangan_SN': bobot[kol] * baris[f'NDA_{kol}'],
        })
    rincian_df = pd.DataFrame(rincian).set_index('kriteria')
    print(f'[{key}] {username} -- AS={baris["AS"]:.4f}, rank={int(baris["rank"])}')
    print(rincian_df.round(4).to_string())
    print(f'Total SP={baris["SP"]:.4f}  (kriteria penyumbang SP terbesar: '
          f'{rincian_df["sumbangan_SP"].idxmax()})')
    return rincian_df


_ = bedah_kontribusi('3class__min30__C5A', 'farhanzubedi')


farhanzubedi tidak ada di kombinasi 3class__min30__C5A (mungkin kefilter).


In [19]:
import pandas as pd
df_check = pd.read_csv(r"C:\Users\nesha\Tugas_akhir\seleksi-influencer\01_pengumpulan_data\04_1_mendapatkan_metadata_akun\video_views.csv")
print(len(df_check), 'baris total')
print(df_check['username'].nunique(), 'username unik')
print(df_check.groupby('username').size().sort_values())

628 baris total
41 username unik
username
farhanzubedi              3
nisaobgyn                 5
drgfira                   6
putriemje                 7
dokternadiyah             8
dokter.reni.anak          9
dr.deamaulidia            9
dr.ndreamon              11
fauzaannibras            12
wishdrboyke              12
meta.hanindita           13
zickyneuro               13
guru.bidan.wulan         14
reisabrotoasmoro         14
roozzaak                 14
itsdoktermuda            14
bidancynthia             15
doktervito               15
dokterprasadja           16
drluckyspa               16
gelarsramdhani           17
dr.saddam_ismail         17
jamilatussadiyah         17
dr.mariojohan            17
tirtacipeng              19
dr.ziee                  19
drhans_spesialisanak     19
dr.zulvia.syarif.spkj    19
drkevinmak               19
dokter.harmonospanak     19
dokteramiraobgyn         20
aymanalts                20
ikhsanqthii              20
giapratamamd             20
ekidar

In [20]:
kolom_c1 = SKEMA_C1['3class']
kolom_p = 'P_' + kolom_c1.replace('C1_', '')

df_debug = skor_influencer[['username', kolom_c1, kolom_p]].merge(metrik_akun, on='username', how='inner')
print('Total awal                         :', len(df_debug))
print('Followers >= 10000                 :', (df_debug['followers'] >= MIN_FOLLOWERS).sum())
print('Jumlah video disentimenin >= 10     :', (df_debug['jumlah_video'] >= MIN_VIDEO).sum())
print('Jumlah komentar (P) >= 30           :', (df_debug[kolom_p] >= 30).sum())
print('Jumlah video ada views >= 10         :', (df_debug['jumlah_video_views'] >= MIN_VIDEO_VIEWS).sum())
print('C3 not NaN                          :', df_debug['C3_rata_views'].notna().sum())
print()
print('Lolos SEMUA filter                  :',
      ((df_debug['followers'] >= MIN_FOLLOWERS)
       & (df_debug['jumlah_video'] >= MIN_VIDEO)
       & (df_debug[kolom_p] >= 30)
       & (df_debug['jumlah_video_views'] >= MIN_VIDEO_VIEWS)
       & (df_debug['C3_rata_views'].notna())).sum())

Total awal                         : 42
Followers >= 10000                 : 40
Jumlah video disentimenin >= 10     : 35
Jumlah komentar (P) >= 30           : 40
Jumlah video ada views >= 10         : 5
C3 not NaN                          : 27

Lolos SEMUA filter                  : 4


In [24]:
overlap_mentah = {}
for u in df_video['username'].unique():
    set_meta = set(df_video[(df_video['username'] == u) & (df_video['video_id'].isin(video_id_disentimenkan))]['video_id'])
    set_views = set(df_views[df_views['username'] == u]['video_id'])
    overlap_mentah[u] = len(set_meta & set_views)

overlap_df = pd.DataFrame({
    'overlap_mentah': pd.Series(overlap_mentah),
}).join(video_agg_views.set_index('username')['jumlah_video_views'], how='left')

overlap_df['selisih'] = overlap_df['overlap_mentah'] - overlap_df['jumlah_video_views'].fillna(0)
print(overlap_df[overlap_df['selisih'] != 0].sort_values('selisih', ascending=False))

                       overlap_mentah  jumlah_video_views  selisih
aliphildan                         20                 NaN     20.0
aymanalts                          20                 NaN     20.0
dokteramiraobgyn                   20                 NaN     20.0
dokter_ika                         20                 NaN     20.0
giapratamamd                       20                 NaN     20.0
doktermedok                        20                 NaN     20.0
ikhsanqthii                        20                 NaN     20.0
dr.ziee                            19                 NaN     19.0
ekidarehanf                        20                 1.0     19.0
tirtacipeng                        19                 NaN     19.0
drkevinmak                         19                 1.0     18.0
dionharyadi                        20                 4.0     16.0
dr.zulvia.syarif.spkj              19                 3.0     16.0
jiemiardian                        20                 5.0     